# 122_autogluon_on_89col_pool (Colab版)

AutoGluonにはこれまで常にFULL 441/447列を投入してきたが、93_のTabPFN/CatBoostで最良点だった89列(TOP20PCT選択)を初めて投入する実験。num_bag_sets=3(104_と同一設定)+predict_proba_oof()による最初からのOOF抽出(108_の技術)を組み込み済み。GPU Runtime必須、TIME_LIMIT=14400秒(4時間)。

In [1]:
!pip install -q catboost autogluon.tabular


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 514.8/514.8 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.4/248.4 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.3/112.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.7/90.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 9.1 MB/s eta 0:00:00


In [2]:
"""122_autogluon_on_89col_pool

これまでAutoGluonには常に441列(44_〜64_世代)または447列(102_/104_、77_パイプライン)の
FULL構成しか与えたことがなく、TabPFN/CatBoostにとって列数チューニングの最適点と判明
済みの89列（93_のTOP20PCT_nested、TabPFN単体でCatBoostを初めて上回った構成、
[[tabpfn-ensemble-partner]]）を与えたことは一度もない。AutoGluonの内部bagging+stacking
という機構を、既に厳選済みの高品質な特徴量空間に適用するという、機構×特徴量プールの
未試行の組み合わせを検証する。

`93_`と同一の重要度選択（上位20%、ただしAutoGluonへの入力を作るだけなので選択ステップは
Train全件で1回だけ行う、93_の"提出用"ステップと同じ設計——ネスト選択は評価用OOFの
リーク対策であり、AutoGluon自身が内部でbaggingによる正しい検証を行うため不要）で
447列→89列に絞り、AutoGluonに投入する。

設定は`104_`（num_bag_sets=3修正版、GPU）を踏襲: presets=best_quality,
excluded_model_types=["FASTAI","NN_TORCH","KNN"], num_bag_folds=8, num_bag_sets=3,
num_stack_levels=1, dynamic_stacking=False, num_gpus=1, TIME_LIMIT=14400秒。

`108_`で確立した`predict_proba_oof()`（再学習不要でOOFを取得できる公式API）をこのスクリプト
に最初から組み込み、OOF・Test予測の両方を1回のColab実行で取得する。93_のbuild_features()は
入社日でソートしない（部署頻度encodingのみtarget依存、hire-date sortが不要）ため、
AutoGluonに渡すフレームの行順は最初からtrain_ids/test_idsの元CSV順と一致しており、
108_で必要だった行順の復元作業は不要。

比較対象: 93_のTabPFN89(Public 0.504065)・CB89(局所val=0.518415)・旧プールC(Public 0.514050)。
このAutoGluon×89列プールのOOFが115_のスタックに有望な追加候補になるかを検証する
（別スクリプトで実施予定）。

出力: weighted / best_single の2種類（Test予測+OOF、.npyで保存）
"""
import datetime
import re
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import catboost as cb
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))
from common.utils.logger import get_logger
from common.utils.seed import seed_everything

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

SCRIPT_NAME = "122_autogluon_on_89col_pool"
TODAY = datetime.datetime.now().strftime("%Y%m%d")
LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values
logger.info(f"Train Persona: {train_persona.shape}, Test Persona: {test_persona.shape}")

EARLY_LEAVER_IDS = set(train_monthly.loc[train_monthly["月末在籍状態"] == "退職", ID_COL].unique())
_test_early = set(test_monthly.loc[test_monthly["月末在籍状態"] == "退職", ID_COL].unique())
assert len(_test_early) == 0

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]


Mounted at /content/drive
[2026-09-06 09:10:59] [INFO] === [122_autogluon_on_89col_pool] 実験開始 ===


INFO:122_autogluon_on_89col_pool:=== [122_autogluon_on_89col_pool] 実験開始 ===


[2026-09-06 09:11:06] [INFO] Train Persona: (2761, 20), Test Persona: (2502, 19)


INFO:122_autogluon_on_89col_pool:Train Persona: (2761, 20), Test Persona: (2502, 19)


## 54_l2_m_interaction.ipynb と同一の特徴量関数（split非依存、84_と同一）

In [3]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]
            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan
            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )
            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i - 1]) and values[i] != values[i - 1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)
        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan
        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan
        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)


def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=42):
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)
    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)
    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)
    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out


def create_quarterly_features(monthly_df, employee_ids, metrics, suffix=""):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean{suffix}"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration{suffix}"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)


def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    if m:
        return m.group(1).strip()
    lines = [l for l in text.strip().splitlines() if re.search(r"勤務地|転居|在宅勤務", l)]
    return "".join(lines) if lines else None


NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")


def classify_reloc(s):
    if s is None:
        return None
    if NEG_RELOC.search(s):
        return False
    if POS_RELOC.search(s):
        return True
    return None


def extract_desired_location_v2(s):
    if s is None:
        return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    loc = m.group(1) if m else None
    if loc is None:
        m2 = re.search(r"(.+?)を希望勤務地", s)
        loc = m2.group(1) if m2 else None
    if loc is None:
        m3 = re.search(r"^([一-龥ぁ-んァ-ンー]+?)(?:での勤務|勤務)?を希望。", s)
        loc = m3.group(1) if m3 else None
    if loc is not None:
        loc = loc.strip("「」")
    return loc


def create_relocation_mismatch_features(persona_df, extract_fn, state_col, flag_col):
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    reloc_ok_raw = ws_section.apply(classify_reloc)
    desired = ws_section.apply(extract_fn)
    actual = persona_df["初期勤務地"]
    match = (desired == actual) & desired.notna()
    reloc_true = reloc_ok_raw == True
    reloc_false = reloc_ok_raw == False
    valid = desired.notna() & reloc_ok_raw.notna()
    state = pd.Series("unknown", index=persona_df.index)
    state[valid & reloc_true & match] = "許容_一致"
    state[valid & reloc_true & ~match] = "許容_不一致"
    state[valid & reloc_false & match] = "非許容_一致"
    state[valid & reloc_false & ~match] = "非許容_不一致"
    double_bad = (valid & reloc_false & ~match).astype(int)
    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        state_col: state.values,
        flag_col: double_bad.values,
    })


_ANALYTICAL_MAJOR = {"情報", "理工学"}
_ANALYTICAL_JOB = {"IT・エンジニアリング", "データ・商品企画・コンサルティング"}


def create_l2_m_interaction_features(persona_df, reloc_v2_df):
    is_analytical_major = persona_df["専攻分野"].isin(_ANALYTICAL_MAJOR)
    is_analytical_job = persona_df["初期職種"].isin(_ANALYTICAL_JOB)
    m_bad = (~is_analytical_major & is_analytical_job).astype(int)
    state = reloc_v2_df.set_index("社員ID").loc[persona_df["社員ID"], "転居x勤務地_状態_v2"].values
    l2_bad = (state == "非許容_不一致").astype(int)
    both_bad = (l2_bad & m_bad)
    risk_count = l2_bad + m_bad
    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        "M_不適合": m_bad,
        "L2xM_ダブル不適合": both_bad,
        "L2xM_リスク要因数": risk_count,
    })


def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()
    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]
    train_te = np.full(len(train_persona), global_mean)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values
    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()
    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values
    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values
    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def create_last_month_category_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        last = emp_data.iloc[-1]
        features_list.append({
            "社員ID": employee_id,
            "最終月の職種": last["職種"],
            "最終月の勤務地": last["勤務地"],
            "最終月の部署ID": last["部署ID"],
        })
    return pd.DataFrame(features_list)


logger.info("=" * 60)
logger.info("split非依存の基本特徴量を生成中...")
train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)
train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)
train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)
train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)
train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)
train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)
train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)
train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

tfidf_train_list, tfidf_test_list = [], []
for col in TEXT_COLS:
    tr, te = create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=SEED)
    tfidf_train_list.append(tr)
    tfidf_test_list.append(te)

D_EXPANDED_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]
train_quarterly_exp = create_quarterly_features(train_monthly, train_ids, D_EXPANDED_METRICS, suffix="_exp")
test_quarterly_exp = create_quarterly_features(test_monthly, test_ids, D_EXPANDED_METRICS, suffix="_exp")

train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])
for col in TEXT_COLS:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)
train_persona["text_total_chars"] = train_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter
train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]
train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

train_reloc_v2 = create_relocation_mismatch_features(train_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")
test_reloc_v2 = create_relocation_mismatch_features(test_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")
train_l2m = create_l2_m_interaction_features(train_persona, train_reloc_v2)
test_l2m = create_l2_m_interaction_features(test_persona, test_reloc_v2)
logger.info("split非依存の基本特徴量生成完了")

logger.info("77_で確認済みの最終月の職種/勤務地/部署ID(3列)を生成中...")
train_lastmonth_cat = create_last_month_category_features(train_monthly, train_ids)
test_lastmonth_cat = create_last_month_category_features(test_monthly, test_ids)

[2026-09-06 09:11:06] [INFO] ============================================================


INFO:122_autogluon_on_89col_pool:============================================================


[2026-09-06 09:11:06] [INFO] split非依存の基本特徴量を生成中...


INFO:122_autogluon_on_89col_pool:split非依存の基本特徴量を生成中...


[2026-09-06 09:22:31] [INFO] split非依存の基本特徴量生成完了


INFO:122_autogluon_on_89col_pool:split非依存の基本特徴量生成完了


[2026-09-06 09:22:31] [INFO] 77_で確認済みの最終月の職種/勤務地/部署ID(3列)を生成中...


INFO:122_autogluon_on_89col_pool:77_で確認済みの最終月の職種/勤務地/部署ID(3列)を生成中...


## build_features(train_id_subset): 80_/81_/82_/83_/84_と同一パターン（純77_相当447列）

In [4]:
def build_features(train_id_subset):
    train_id_subset = set(train_id_subset)
    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_id_subset, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")
    tf = tf.merge(train_quarterly_exp, on=ID_COL, how="left")
    for trdf in tfidf_train_list:
        tf = tf.merge(trdf, on=ID_COL, how="left")
    tf = tf.merge(train_reloc_v2, on=ID_COL, how="left")
    tf = tf.merge(train_l2m, on=ID_COL, how="left")
    tf = tf.merge(train_lastmonth_cat, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")
    ttf = ttf.merge(test_quarterly_exp, on=ID_COL, how="left")
    for tedf in tfidf_test_list:
        ttf = ttf.merge(tedf, on=ID_COL, how="left")
    ttf = ttf.merge(test_reloc_v2, on=ID_COL, how="left")
    ttf = ttf.merge(test_l2m, on=ID_COL, how="left")
    ttf = ttf.merge(test_lastmonth_cat, on=ID_COL, how="left")

    _train_period_features = tf[tf[ID_COL].isin(train_id_subset)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df_ in [tf, ttf]:
        for m in job_dev_metrics:
            df_[f"{m}_job_deviation"] = df_[m] - df_["初期職種"].map(job_means[m])
        df_["研修時間_職種比"] = df_["研修時間_mean"] / df_["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df_["研修時間_区分比"] = df_["研修時間_mean"] / df_["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df_["初任給_等級内偏差"] = df_["初任給_円"] - df_["初期等級"].map(grade_salary_mean)
        df_["初任給_区分内偏差"] = df_["初任給_円"] - df_["入社区分"].map(category_salary_mean)
        df_["月例給与_等級内偏差"] = df_["月例給与_円_mean"] - df_["初期等級"].map(grade_monthly_salary_mean)

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)
    tf[TARGET_COL] = train_persona.set_index(ID_COL).loc[tf.index, TARGET_COL].values
    return tf, ttf


def _feature_cols(df):
    return [c for c in df.columns if c not in ["入社日", TARGET_COL]]

A_PARAMS = {
    "depth": 4,
    "learning_rate": 0.03518359458951149,
    "l2_leaf_reg": 2.217690447016724,
    "border_count": 218,
    "bagging_temperature": 0.6787467566574921,
    "random_strength": 1.438494697238285,
}
ITER = 560
OOF_SEED = 42
TOP_PCT_20 = 0.2

surv_mask = np.array([tid not in EARLY_LEAVER_IDS for tid in train_ids])
logger.info(f"生存者(24か月在籍): {surv_mask.sum()} / {len(surv_mask)}")


def _fit_one_classifier(X_tr, y_tr, obj_cols, seed):
    model = cb.CatBoostClassifier(**A_PARAMS, iterations=ITER, random_seed=seed,
                                   verbose=False, cat_features=obj_cols, task_type="CPU")
    model.fit(X_tr, y_tr)
    return model


def select_top_pct(imp_series, pct):
    n_top = max(1, round(len(imp_series) * pct))
    return imp_series.sort_values(ascending=False).head(n_top).index.tolist()


def save_submission(preds, config_label):
    path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}_submission.csv"
    pd.DataFrame({ID_COL: test_ids, TARGET_COL: preds}).to_csv(path, index=False, header=False)
    logger.info(f"  提出ファイル保存: {path.name}（予測平均={preds.mean():.4f}）")
    return str(path)

[2026-09-06 09:23:03] [INFO] 生存者(24か月在籍): 2632 / 2761


INFO:122_autogluon_on_89col_pool:生存者(24か月在籍): 2632 / 2761


## Train全件のtf/ttf(447列)を構築し、93_と同じ重要度上位20%選択でAutoGluon投入用89列を作る

In [5]:
logger.info("=" * 60)
logger.info("Train全件での447列フレームを構築中...")
tf_full, ttf_full = build_features(train_ids.tolist())
FULL_FEATURE_COLS = _feature_cols(tf_full)
assert len(FULL_FEATURE_COLS) == 447, f"想定外の列数: {len(FULL_FEATURE_COLS)}"

obj_cols_full = [c for c in FULL_FEATURE_COLS if tf_full[c].dtype == "object"]
X_tr_full = tf_full[FULL_FEATURE_COLS].fillna(-999)
y_tr_full = tf_full[TARGET_COL]

m_selector_full = _fit_one_classifier(X_tr_full, y_tr_full, obj_cols_full, OOF_SEED)
imp_series_full = pd.Series(m_selector_full.get_feature_importance(), index=FULL_FEATURE_COLS)
TOP20_COLS = select_top_pct(imp_series_full, TOP_PCT_20)
logger.info(f"TOP20PCT選択: {len(TOP20_COLS)}列（93_と同一の選択のはず）")

[2026-09-06 09:23:03] [INFO] ============================================================


INFO:122_autogluon_on_89col_pool:============================================================


[2026-09-06 09:23:03] [INFO] Train全件での447列フレームを構築中...


INFO:122_autogluon_on_89col_pool:Train全件での447列フレームを構築中...


[2026-09-06 09:23:20] [INFO] TOP20PCT選択: 89列（93_と同一の選択のはず）


INFO:122_autogluon_on_89col_pool:TOP20PCT選択: 89列（93_と同一の選択のはず）


## AutoGluon投入用フレーム(89列、行順はtrain_ids/test_idsの元CSV順のまま—— build_features()は入社日でソートしないため、108_のような行順復元は不要)

In [6]:
ag_full = tf_full[TOP20_COLS + [TARGET_COL]].reset_index(drop=True)
ag_test = ttf_full[TOP20_COLS].reset_index(drop=True)
logger.info(f"ag_full: {ag_full.shape} / ag_test: {ag_test.shape}")

[2026-09-06 09:23:20] [INFO] ag_full: (2761, 90) / ag_test: (2502, 89)


INFO:122_autogluon_on_89col_pool:ag_full: (2761, 90) / ag_test: (2502, 89)


## AutoGluon（104_と同一設定、num_bag_sets=3・GPU）

In [7]:
logger.info("=" * 60)
logger.info("AutoGluon (autogluon.tabular) を読み込み中...")
from autogluon.tabular import TabularPredictor

try:
    import ray  # noqa: F401
    logger.warning("⚠️ ray が入っている。ParallelLocalFoldFittingStrategy経由でfold全滅の危険。")
except ImportError:
    logger.info("✅ ray は入っていない（正常）。foldは逐次学習される。")

import torch
HAS_GPU = torch.cuda.is_available()
logger.info(f"GPU利用可能: {HAS_GPU}")
if not HAS_GPU:
    logger.warning("GPUが無い。ColabのランタイムをGPUに変更すること。")

PRESETS = "best_quality"
TIME_LIMIT = 14400
AG_METRIC = "log_loss"
EXCLUDED_MODELS = ["FASTAI", "NN_TORCH", "KNN"]
DYNAMIC_STACKING = False
NUM_STACK_LEVELS = 1
NUM_BAG_FOLDS = 8
NUM_BAG_SETS = 3
NUM_GPUS = 1 if HAS_GPU else 0

AG_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
AG_DIR.mkdir(parents=True, exist_ok=True)


def fit_autogluon(tag, train_df, time_limit=TIME_LIMIT, num_gpus=NUM_GPUS):
    path = AG_DIR / tag
    if (path / "predictor.pkl").exists():
        logger.info(f"[{tag}] 保存済みpredictorを読み込む（再学習しない）")
        return TabularPredictor.load(str(path))

    logger.info("=" * 60)
    logger.info(f"[{tag}] AutoGluon fit: n={len(train_df)}, 特徴量={len(TOP20_COLS)}, "
                f"time_limit={time_limit}, num_gpus={num_gpus}, "
                f"num_bag_folds={NUM_BAG_FOLDS}, num_bag_sets={NUM_BAG_SETS}")
    kw = dict(presets=PRESETS, time_limit=time_limit,
              excluded_model_types=EXCLUDED_MODELS,
              num_bag_folds=NUM_BAG_FOLDS, num_bag_sets=NUM_BAG_SETS,
              num_stack_levels=NUM_STACK_LEVELS,
              dynamic_stacking=DYNAMIC_STACKING, num_gpus=num_gpus)
    p = TabularPredictor(label=TARGET_COL, eval_metric=AG_METRIC, path=str(path),
                         problem_type="binary")
    p.fit(train_df, **kw)
    return p


def leaderboard(predictor):
    try:
        return predictor.leaderboard(silent=True)
    except TypeError:
        return predictor.leaderboard()


def positive_proba(predictor, X, model=None):
    pp = predictor.predict_proba(X, model=model)
    pos = predictor.positive_class if hasattr(predictor, "positive_class") else None
    if pos is None or pos not in pp.columns:
        pos = 1 if 1 in pp.columns else pp.columns[-1]
    return pp[pos].values if hasattr(pp, "columns") else pp

[2026-09-06 09:23:20] [INFO] ============================================================


INFO:122_autogluon_on_89col_pool:============================================================


[2026-09-06 09:23:20] [INFO] AutoGluon (autogluon.tabular) を読み込み中...


INFO:122_autogluon_on_89col_pool:AutoGluon (autogluon.tabular) を読み込み中...


[2026-09-06 09:23:20] [INFO] ✅ ray は入っていない（正常）。foldは逐次学習される。


INFO:122_autogluon_on_89col_pool:✅ ray は入っていない（正常）。foldは逐次学習される。


[2026-09-06 09:23:20] [INFO] GPU利用可能: True


INFO:122_autogluon_on_89col_pool:GPU利用可能: True


## 実行(チェックポイントあり、Colab切断時も再開可能)

In [8]:
logger.info("=" * 60)
predictor = fit_autogluon("full89", ag_full)
lb = leaderboard(predictor)
logger.info(f"\n===== leaderboard(AutoGluon内部検証) =====\n"
            f"{lb[['model', 'score_val', 'fit_time']].head(15).to_string(index=False)}")

weighted_names = lb[lb["model"].str.startswith("WeightedEnsemble")]["model"].tolist()
single_names = lb[~lb["model"].str.startswith("WeightedEnsemble")]["model"].tolist()

[2026-09-06 09:23:20] [INFO] ============================================================


INFO:122_autogluon_on_89col_pool:============================================================


[2026-09-06 09:23:20] [INFO] ============================================================


INFO:122_autogluon_on_89col_pool:============================================================


[2026-09-06 09:23:20] [INFO] [full89] AutoGluon fit: n=2761, 特徴量=89, time_limit=14400, num_gpus=1, num_bag_folds=8, num_bag_sets=3


...

[1000]	valid_set's binary_logloss: 0.465202


[1000]	valid_set's binary_logloss: 0.431626


	-0.4763	 = Validation score   (-log_loss)
	467.76s	 = Training   runtime
	1.25s	 = Validation runtime
Fitting model: RandomForest_r39_BAG_L2 ... Training model for up to 735.33s of the 735.17s of remaining time.
	Fitting 1 model on all data (use_child_oof=True) | Fitting with cpus=2, gpus=0, mem=0.0/9.7 GB
	-0.4916	 = Validation score   (-log_loss)
	49.75s	 = Training   runtime
	0.22s	 = Validation runtime
Fitting model: CatBoost_r167_BAG_L2 ... Training model for up to 685.20s of the 685.04s of remaining time.
	Fitting 24 child models (S1F1 - S3F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=1, gpus=0)
	Training S1F1 with GPU, note that this may negatively impact model quality compared to CPU training.
	Training S1F2 with GPU, note that this may negatively impact model quality compared to CPU training.
	Training S1F3 with GPU, note that this may negatively impact model quality compared to CPU training.
	Training S1F4 with GPU, note that this may negatively imp

[2026-09-06 13:23:22] [INFO] 
===== leaderboard(AutoGluon内部検証) =====
               model  score_val    fit_time
 WeightedEnsemble_L3  -0.464678 2022.184269
CatBoost_r177_BAG_L2  -0.465109 1700.735203
CatBoost_r167_BAG_L2  -0.465574 1784.856535
 CatBoost_r50_BAG_L2  -0.465942 1659.007094
 CatBoost_r13_BAG_L2  -0.466089 2120.071461
 CatBoost_r69_BAG_L2  -0.467307 1740.008132
     CatBoost_BAG_L2  -0.467427 1766.707583
CatBoost_r137_BAG_L2  -0.467440 1728.233733
 CatBoost_r70_BAG_L2  -0.467552 1687.812373
  CatBoost_r9_BAG_L2  -0.468800 1726.474102
 LightGBM_r15_BAG_L2  -0.469388 1689.902688
   LightGBMXT_BAG_L2  -0.470739 1665.680939
LightGBM_r130_BAG_L2  -0.471073 1665.288943
  XGBoost_r89_BAG_L2  -0.471237 1598.108435
 LightGBM_r96_BAG_L2  -0.471879 1643.730492


INFO:122_autogluon_on_89col_pool:
===== leaderboard(AutoGluon内部検証) =====
               model  score_val    fit_time
 WeightedEnsemble_L3  -0.464678 2022.184269
CatBoost_r177_BAG_L2  -0.465109 1700.735203
CatBoost_r167_BAG_L2  -0.465574 1784.856535
 CatBoost_r50_BAG_L2  -0.465942 1659.007094
 CatBoost_r13_BAG_L2  -0.466089 2120.071461
 CatBoost_r69_BAG_L2  -0.467307 1740.008132
     CatBoost_BAG_L2  -0.467427 1766.707583
CatBoost_r137_BAG_L2  -0.467440 1728.233733
 CatBoost_r70_BAG_L2  -0.467552 1687.812373
  CatBoost_r9_BAG_L2  -0.468800 1726.474102
 LightGBM_r15_BAG_L2  -0.469388 1689.902688
   LightGBMXT_BAG_L2  -0.470739 1665.680939
LightGBM_r130_BAG_L2  -0.471073 1665.288943
  XGBoost_r89_BAG_L2  -0.471237 1598.108435
 LightGBM_r96_BAG_L2  -0.471879 1643.730492


## OOF抽出(108_で確立したpredict_proba_oof、再学習不要) + Test予測

In [9]:
logger.info("=" * 60)
logger.info("predict_proba_oof()でOOF予測を取得中(再学習なし)...")

results = {}
for kind, names in [("weighted", weighted_names), ("best_single", single_names)]:
    if not names:
        logger.info(f"  [{kind}] 該当モデルなし。スキップ")
        continue
    model_name = names[0]
    oof_df = predictor.predict_proba_oof(model=model_name, as_multiclass=True)
    pos_class = predictor.positive_class if hasattr(predictor, "positive_class") else 1
    if pos_class not in oof_df.columns:
        pos_class = 1 if 1 in oof_df.columns else oof_df.columns[-1]
    oof_values = oof_df[pos_class].values
    assert len(oof_values) == len(train_ids), (
        f"[{kind}] OOF行数({len(oof_values)})がtrain_ids数({len(train_ids)})と一致しない"
    )
    oof_score = log_loss(y_train.values[surv_mask], oof_values[surv_mask])
    logger.info(f"  [{kind}] model={model_name} / OOF val(生存者)={oof_score:.6f}")

    test_pred = positive_proba(predictor, ag_test, model=model_name)
    path = save_submission(test_pred, kind)
    np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{kind}_oofpreds.npy", oof_values)
    np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{kind}_testpreds.npy", test_pred)
    results[kind] = (oof_score, path)

logger.info("=" * 60)
logger.info("=== 結果まとめ ===")
for kind, (oof_score, path) in results.items():
    logger.info(f"  {kind}: OOF val(生存者)={oof_score:.6f} / {path}")
logger.info("(参考: 93_のTabPFN89 Public=0.504065、局所val=0.503880 / CB89局所val=0.518415 / "
            "旧プールC(441列8本平均) Public=0.514050)")
logger.info(f"=== [{SCRIPT_NAME}] 実験終了 ===")

[2026-09-06 13:23:22] [INFO] ============================================================


INFO:122_autogluon_on_89col_pool:============================================================


[2026-09-06 13:23:22] [INFO] predict_proba_oof()でOOF予測を取得中(再学習なし)...


INFO:122_autogluon_on_89col_pool:predict_proba_oof()でOOF予測を取得中(再学習なし)...


[2026-09-06 13:23:22] [INFO]   [weighted] model=WeightedEnsemble_L3 / OOF val(生存者)=0.483221


...

INFO:122_autogluon_on_89col_pool:  best_single: OOF val(生存者)=0.483859 / /content/drive/MyDrive/jaggle_2026/data/output/20260906/20260906_122_autogluon_on_89col_pool_best_single_submission.csv


[2026-09-06 13:23:47] [INFO] (参考: 93_のTabPFN89 Public=0.504065、局所val=0.503880 / CB89局所val=0.518415 / 旧プールC(441列8本平均) Public=0.514050)


INFO:122_autogluon_on_89col_pool:(参考: 93_のTabPFN89 Public=0.504065、局所val=0.503880 / CB89局所val=0.518415 / 旧プールC(441列8本平均) Public=0.514050)


[2026-09-06 13:23:47] [INFO] === [122_autogluon_on_89col_pool] 実験終了 ===


INFO:122_autogluon_on_89col_pool:=== [122_autogluon_on_89col_pool] 実験終了 ===
